# 04 — Feature Selection

**Objetivo:** Selecionar as features mais relevantes para o modelo preditivo de turnover voluntário,
aplicando 3 camadas de filtro progressivo. Todos os filtros são calculados **exclusivamente no treino**.

**Referência SPEC:** Seção 7

---

| Entrada | Descrição |
|---|---|
| `data/processed/splits/train.parquet` | Split de treino (70% dos IDs) |
| `data/processed/splits/val.parquet` | Split de validação |
| `data/processed/splits/test.parquet` | Split de teste |
| `data/processed/splits/scaler_cols.json` | Metadados dos splits |

| Saída | Descrição |
|---|---|
| `data/processed/feature_selection/selected_features.json` | Lista final de features selecionadas |
| `data/processed/feature_selection/fs_report.parquet` | Relatório completo por feature (todas as camadas) |
| `reports/feature_selection/fs_layer1_dropped.txt` | Features removidas na Camada 1 |
| `reports/feature_selection/fs_layer2_rfecv.png` | Curva RFECV |
| `reports/feature_selection/fs_layer3_shap.png` | SHAP summary plot |

---

### Camadas de Feature Selection

**Camada 1 — Filtros de qualidade** (sem modelo):
1. Variância zero (ou near-zero) → drop
2. Missing > 20% → drop *(já garantido no 02, mas verificar nos desligados do protótipo)*
3. Correlação > 0.85 entre features → drop a com mais missing (empate: drop a última alfabeticamente)

**Camada 2 — RFECV com preferência por categorizadas** (SPEC §7):
- Estimador: XGBoost
- Métrica: ROC-AUC
- CV: StratifiedKFold (k=5)
- Preferência por features `_bin` sobre contínuas quando ambas representam a mesma variável original

**Camada 3 — SHAP pós-treino**:
- XGBoost fitado com as features da Camada 2
- Features com SHAP mean |importance| = 0 → drop
- Gera summary plot ranqueado

## 1 · Setup & Configuração

In [6]:
import pandas as pd
import numpy as np
import json
import joblib
import warnings
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
from pathlib import Path
from datetime import datetime

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import RFECV
from xgboost import XGBClassifier
import shap

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

print(f"pandas {pd.__version__}  ·  numpy {np.__version__}")
print(f"Execução: {datetime.now():%Y-%m-%d %H:%M}")

pandas 2.3.3  ·  numpy 2.3.5
Execução: 2026-04-27 15:55


In [7]:
# ════════════════════════════════════════════════════════════════
# CONFIGURAÇÃO
# ════════════════════════════════════════════════════════════════
PROJECT_ROOT    = Path.cwd().parent
DATA_SPLITS     = PROJECT_ROOT / "data" / "processed" / "splits"   # base; subpastas por grupo
FS_DIR          = PROJECT_ROOT / "data" / "processed" / "feature_selection"  # base; subpastas por grupo
REPORTS_FS      = PROJECT_ROOT / "reports" / "feature_selection"   # base; subpastas por grupo

# ── Grupos operacionais ───────────────────────────────────────
# A feature selection é executada de forma independente por grupo:
#   splits/{grupo}/ → feature_selection/{grupo}/
# Isso garante que cada grupo tenha suas próprias features selecionadas,
# respeitando as particularidades de cada população de colaboradores.
GRUPOS = ["Vendas", "Transporte", "Fábrica"]

# ── Parâmetros ────────────────────────────────────────────────
RNG_SEED            = 42
CORR_THRESHOLD      = 0.85   # Camada 1: correlação máxima entre features
NULL_THRESHOLD      = 0.20   # Camada 1: máximo de missing aceitável
RFECV_CV            = 5      # Camada 2: folds do StratifiedKFold
RFECV_STEP          = 5      # Camada 2: features removidas por iteração
RFECV_MIN_FEATURES  = 10     # Camada 2: mínimo de features a manter

print(f"Corr. threshold   : {CORR_THRESHOLD}")
print(f"Null threshold    : {NULL_THRESHOLD:.0%}")
print(f"RFECV CV folds    : {RFECV_CV}")
print(f"Grupos            : {GRUPOS}")


Corr. threshold   : 0.85
Null threshold    : 20%
RFECV CV folds    : 5
Grupos            : ['Vendas', 'Transporte', 'Fábrica']


## 2 · Carga dos Splits

In [ ]:

# ── Feature selection por grupo operacional ──────────────────
# Para cada grupo: carrega os splits de splits/{grupo}/, executa as 3 camadas
# de seleção e salva os resultados em feature_selection/{grupo}/.

for GRUPO in GRUPOS:
    print(f"\n{'='*65}")
    print(f"GRUPO: {GRUPO}")
    print('='*65)

    # ── Diretórios do grupo ───────────────────────────────────
    grupo_splits  = DATA_SPLITS / GRUPO
    grupo_fs_dir  = FS_DIR      / GRUPO
    grupo_reports = REPORTS_FS  / GRUPO
    grupo_fs_dir.mkdir(parents=True, exist_ok=True)
    grupo_reports.mkdir(parents=True, exist_ok=True)

    # ── Carregar metadados dos splits ─────────────────────────
    scaler_info = json.loads((grupo_splits / "scaler_cols.json").read_text(encoding="utf-8"))
    COL_ID      = scaler_info["col_id"]
    COL_TARGET  = scaler_info["col_target"]
    COL_DATA    = scaler_info["col_data"]

    # ── Carregar splits ───────────────────────────────────────
    df_train = pd.read_parquet(grupo_splits / "train.parquet")
    df_val   = pd.read_parquet(grupo_splits / "val.parquet")

    META = {COL_ID, COL_DATA, COL_TARGET}
    # Excluir todas as variantes de fg_status_vol — são output/target, nunca features
    META.update(c for c in df_train.columns if c.startswith("fg_status_vol"))
    META.update(c for c in df_train.columns if c.startswith("fg_demitido_voluntario"))
    for c in ["ds_grupo", "agrupamento_final"]:
        if c in df_train.columns:
            META.add(c)

    fg_status_leaked = [c for c in df_train.columns
                        if c.startswith("fg_status_vol") and c != COL_TARGET]
    if fg_status_leaked:
        print(f"  ⚠ {len(fg_status_leaked)} coluna(s) fg_status_vol excluídas (output leakage): {fg_status_leaked}")

    feature_cols = [c for c in df_train.columns if c not in META]
    X_train = df_train[feature_cols].copy()
    y_train = df_train[COL_TARGET].copy()
    X_val   = df_val[feature_cols].copy()
    y_val   = df_val[COL_TARGET].copy()

    print(f"  Treino : {X_train.shape[0]:,} × {X_train.shape[1]}  |  turnover: {y_train.mean():.2%}")
    print(f"  Val    : {X_val.shape[0]:,} × {X_val.shape[1]}  |  turnover: {y_val.mean():.2%}")
    print(f"  Features: {len(feature_cols)}")

    # ── Camada 1 · Filtros de qualidade ──────────────────────
    null_pct = X_train.isna().mean()

    # Filtro 1: Variância zero
    drop_var  = X_train.var()[X_train.var() == 0].index.tolist()

    # Filtro 2: Missing > 20%
    drop_null = null_pct[null_pct > NULL_THRESHOLD].index.tolist()

    print(f"\n  [Camada 1] var_zero={len(drop_var)}  missing>{NULL_THRESHOLD:.0%}={len(drop_null)}")

    drop_layer1_pre    = list(set(drop_var) | set(drop_null))
    features_after_pre = [c for c in feature_cols if c not in drop_layer1_pre]

    # Filtro 3: Alta correlação (> CORR_THRESHOLD)
    X_corr       = X_train[features_after_pre]
    corr_matrix  = X_corr.corr(method="pearson").abs()
    upper        = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_corr_pairs = [
        (col, row, upper.loc[row, col])
        for col in upper.columns
        for row in upper.index
        if pd.notna(upper.loc[row, col]) and upper.loc[row, col] > CORR_THRESHOLD
    ]
    drop_corr = set()
    for col_a, col_b, corr_val in high_corr_pairs:
        if col_a in drop_corr or col_b in drop_corr:
            continue
        null_a = null_pct.get(col_a, 0)
        null_b = null_pct.get(col_b, 0)
        if null_a > null_b:
            drop_corr.add(col_a)
        elif null_b > null_a:
            drop_corr.add(col_b)
        else:
            drop_corr.add(max(col_a, col_b))

    drop_layer1     = list(set(drop_layer1_pre) | drop_corr)
    features_layer1 = [c for c in feature_cols if c not in drop_layer1]

    print(f"  [Camada 1] correlação={len(drop_corr)}  →  {len(features_layer1)} restantes")

    # Salvar log Camada 1
    log_path = grupo_reports / "fs_layer1_dropped.txt"
    with open(log_path, "w", encoding="utf-8") as f:
        f.write(f"=== {GRUPO} — Camada 1 — Features Removidas ({datetime.now():%Y-%m-%d %H:%M})\n\n")
        f.write(f"[Variância zero — {len(drop_var)}]\n")
        for c in sorted(drop_var):   f.write(f"  {c}\n")
        f.write(f"\n[Missing > {NULL_THRESHOLD:.0%} — {len(drop_null)}]\n")
        for c in sorted(drop_null):  f.write(f"  {c}  ({null_pct[c]:.1%})\n")
        f.write(f"\n[Alta correlação (>{CORR_THRESHOLD}) — {len(drop_corr)}]\n")
        for c in sorted(drop_corr):  f.write(f"  {c}\n")
        f.write(f"\nTotal removidas: {len(drop_layer1)}\nRestantes: {len(features_layer1)}\n")

    # ── Camada 2 · RFECV com preferência por _bin ─────────────
    X_l1       = X_train[features_layer1].copy()
    null_check = X_l1.isna().sum().sum()
    if null_check > 0:
        X_l1 = X_l1.fillna(X_l1.median())

    neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
    spw      = neg / pos

    print(f"\n  [Camada 2] RFECV — {len(features_layer1)} features entrada  (pode levar alguns minutos)...")

    estimator = XGBClassifier(
        n_estimators=100, max_depth=4, learning_rate=0.1,
        scale_pos_weight=spw, random_state=RNG_SEED,
        eval_metric="logloss", verbosity=0, n_jobs=-1,
    )
    cv    = StratifiedKFold(n_splits=RFECV_CV, shuffle=True, random_state=RNG_SEED)
    rfecv = RFECV(
        estimator=estimator, step=RFECV_STEP, cv=cv, scoring="roc_auc",
        min_features_to_select=RFECV_MIN_FEATURES, n_jobs=-1,
    )
    rfecv.fit(X_l1, y_train)
    features_rfecv  = X_l1.columns[rfecv.support_].tolist()
    rfecv_best_auc  = float(rfecv.cv_results_["mean_test_score"].max())

    print(f"  [Camada 2] {len(features_rfecv)} selecionadas  |  melhor AUC={rfecv_best_auc:.4f}")

    # Preferência _bin (SPEC §7): se par (contínua, contínua_bin) ambas em RFECV → drop contínua
    rfecv_set     = set(features_rfecv)
    drop_bin_pref = set()
    for col in list(rfecv_set):
        if not col.endswith("_bin") and (col + "_bin") in rfecv_set:
            drop_bin_pref.add(col)
    features_layer2 = [c for c in features_rfecv if c not in drop_bin_pref]
    print(f"  [Camada 2] pref._bin={len(drop_bin_pref)}  →  {len(features_layer2)} restantes")

    # Curva RFECV
    mean_scores = rfecv.cv_results_["mean_test_score"]
    n_grid      = list(range(RFECV_MIN_FEATURES, len(features_layer1) + 1, RFECV_STEP))
    n_grid_len  = min(len(mean_scores), len(n_grid))
    fig, ax     = plt.subplots(figsize=(10, 4))
    ax.plot(n_grid[:n_grid_len], mean_scores[:n_grid_len],
            marker="o", markersize=3, linewidth=1.5, color="steelblue")
    ax.axvline(rfecv.n_features_, color="tomato", linestyle="--",
               label=f"Ótimo: {rfecv.n_features_}")
    ax.set_title(f"RFECV — {GRUPO}"); ax.set_xlabel("N features"); ax.set_ylabel("ROC-AUC (CV)")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(grupo_reports / "fs_layer2_rfecv.png", dpi=150, bbox_inches="tight")
    plt.close()

    # ── Camada 3 · SHAP pós-treino ────────────────────────────
    X_l2     = X_l1[features_layer2].copy()
    X_val_l2 = df_val[features_layer2].copy()

    xgb_shap = XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        scale_pos_weight=spw, random_state=RNG_SEED,
        eval_metric="logloss", verbosity=0, n_jobs=-1,
    )
    xgb_shap.fit(X_l2, y_train, eval_set=[(X_val_l2, y_val)], verbose=False)

    from sklearn.metrics import roc_auc_score
    import xgboost as xgb
    auc_val     = roc_auc_score(y_val, xgb_shap.predict_proba(X_val_l2)[:, 1])
    # Usa SHAP nativo do XGBoost (pred_contribs) — evita dependência da extensão C do shap
    dmat_shap   = xgb.DMatrix(X_l2.values, feature_names=list(X_l2.columns))
    shap_values = xgb_shap.get_booster().predict(dmat_shap, pred_contribs=True)[:, :-1]  # drop bias col
    shap_imp    = pd.Series(
        np.abs(shap_values).mean(axis=0), index=features_layer2
    ).sort_values(ascending=False)

    drop_shap       = shap_imp[shap_imp == 0].index.tolist()
    features_layer3 = shap_imp[shap_imp > 0].index.tolist()

    print(f"\n  [Camada 3] SHAP=0={len(drop_shap)}  →  {len(features_layer3)} FINAIS  |  AUC val={auc_val:.4f}")

    # SHAP summary plot
    top_n    = min(30, len(features_layer3))
    top_cols = shap_imp.head(top_n).index.tolist()
    idx_top  = [features_layer2.index(c) for c in top_cols]
    fig, ax  = plt.subplots(figsize=(10, max(6, top_n * 0.28)))
    shap.summary_plot(
        shap_values[:, idx_top], X_l2[top_cols],
        feature_names=top_cols, max_display=top_n, show=False, plot_size=None,
    )
    plt.title(f"SHAP Summary — {GRUPO} — Top {top_n}", pad=12)
    plt.tight_layout()
    plt.savefig(grupo_reports / "fs_layer3_shap.png", dpi=150, bbox_inches="tight")
    plt.close()

    # ── Relatório consolidado ─────────────────────────────────
    report_rows = []
    for col in feature_cols:
        report_rows.append({
            "feature"         : col,
            "var_zero"        : col in drop_var,
            "missing_alto"    : col in drop_null,
            "drop_correlacao" : col in drop_corr,
            "passou_layer1"   : col in features_layer1,
            "passou_rfecv"    : col in features_rfecv,
            "drop_bin_pref"   : col in drop_bin_pref,
            "passou_layer2"   : col in features_layer2,
            "shap_importance" : float(shap_imp.get(col, 0.0)),
            "drop_shap_zero"  : col in drop_shap,
            "SELECIONADA"     : col in features_layer3,
        })
    df_report = pd.DataFrame(report_rows).sort_values(
        ["SELECIONADA", "shap_importance"], ascending=[False, False]
    ).reset_index(drop=True)

    # ── Salvar outputs ────────────────────────────────────────
    selected_path = grupo_fs_dir / "selected_features.json"
    with open(selected_path, "w", encoding="utf-8") as f:
        json.dump({
            "selected_features": features_layer3,
            "n_features"       : len(features_layer3),
            "col_target"       : COL_TARGET,
            "col_id"           : COL_ID,
            "col_data"         : COL_DATA,
            "grupo"            : GRUPO,
            "rfecv_best_auc"   : rfecv_best_auc,
            "xgb_val_auc"      : float(auc_val),
        }, f, indent=2, ensure_ascii=False)

    report_path = grupo_fs_dir / "fs_report.parquet"
    df_report.to_parquet(report_path, index=False)

    print(f"\n  ✓ selected_features.json  ({len(features_layer3)} features)")
    print(f"  ✓ fs_report.parquet        ({len(df_report)} linhas)")
    print(f"  ✓ Outputs: {grupo_fs_dir.relative_to(PROJECT_ROOT)}")
    print(f"  ✓ Reports: {grupo_reports.relative_to(PROJECT_ROOT)}")

print(f"\n{'='*65}")
print(f"✓ Feature selection concluída para {len(GRUPOS)} grupos: {GRUPOS}")
print(f"{'='*65}")



GRUPO: Vendas
  ⚠ 6 coluna(s) fg_status_vol excluídas (output leakage): ['fg_status_vol_3m', 'fg_status_vol_5m', 'fg_status_vol_6m', 'fg_status_vol_7m', 'fg_status_vol_8m', 'fg_status_vol_9m']
  Treino : 8,873 × 156  |  turnover: 38.77%
  Val    : 5,413 × 156  |  turnover: 15.37%
  Features: 156

  [Camada 1] var_zero=2  missing>20%=0
  [Camada 1] correlação=65  →  89 restantes

  [Camada 2] RFECV — 89 features entrada  (pode levar alguns minutos)...
  [Camada 2] 59 selecionadas  |  melhor AUC=0.8594
  [Camada 2] pref._bin=0  →  59 restantes

  [Camada 3] SHAP=0=0  →  59 FINAIS  |  AUC val=0.8193

  ✓ selected_features.json  (59 features)
  ✓ fs_report.parquet        (156 linhas)
  ✓ Outputs: data\processed\feature_selection\Vendas
  ✓ Reports: reports\feature_selection\Vendas

GRUPO: Transporte
  ⚠ 6 coluna(s) fg_status_vol excluídas (output leakage): ['fg_status_vol_3m', 'fg_status_vol_5m', 'fg_status_vol_6m', 'fg_status_vol_7m', 'fg_status_vol_8m', 'fg_status_vol_9m']
  Treino : 12

## 3 · Camada 1 — Filtros de Qualidade (SPEC §7 Camada 1)

Filtros aplicados sequencialmente:
1. **Variância zero** — colunas constantes ou near-zero variance
2. **Missing > 20%** — verificação extra (já tratado no 02)
3. **Alta correlação** — quando duas features têm correlação > 0,85, remove a que tem mais missing (empate: última alfabeticamente)

In [4]:
# Camada 1 consolidada no loop por grupo acima.


In [74]:
# Camada 1 (filtro correlação) consolidada no loop por grupo acima.


In [75]:
# Salvar log Camada 1 consolidado no loop por grupo acima.


## 4 · Camada 2 — RFECV com Preferência por `_bin` (SPEC §7 Camada 2)

**Preferência por categorizadas (SPEC):** para cada variável original que existe em duas versões
(contínua e `_bin`), prioriza a versão `_bin` — se a versão contínua for eliminada pelo RFECV,
não ressuscita; se a `_bin` sendo parcialmente escolhida, mantém a contínua apenas se a `_bin`
não foi selecionada.

Na prática: roda RFECV normalmente e, ao final, para pares `(feature_X, feature_X_bin)` onde
ambos sobrevivem, mantém apenas a versão `_bin`.

In [76]:
# Camada 2 (RFECV prep) consolidada no loop por grupo acima.


In [77]:
# RFECV consolidado no loop por grupo acima.


In [78]:
# Curva RFECV consolidada no loop por grupo acima.


In [79]:
# Preferência _bin consolidada no loop por grupo acima.


## 5 · Camada 3 — SHAP Pós-Treino (SPEC §7 Camada 3)

In [80]:
# Camada 3 (XGBoost para SHAP) consolidada no loop por grupo acima.


In [81]:
# SHAP values consolidados no loop por grupo acima.


In [82]:
# SHAP plot consolidado no loop por grupo acima.


## 6 · Verificação & Relatório Final

In [83]:
# Relatório consolidado no loop por grupo acima.


In [84]:
# Salvar outputs consolidado no loop por grupo acima.


In [5]:
# Resumo final consolidado no loop por grupo acima.
